In [2]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import re
from scipy.stats import kendalltau, spearmanr

# ============ 配置 ============
client = OpenAI(
    base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
    api_key='ida_TbtBzRyj8HeV7kmxkeRsgImHnYelRIUMQ3vW9VIf'
)

MODEL = 'qwen-2.5-72b-instruct'

# ============ Few-Shot Prompt ============
FEW_SHOT_PROMPT_TEMPLATE = """You are a professional search system optimization expert. Your task is to determine if Pseudo-Relevance Feedback (PRF) should be applied to expand a query.

Here are some reference examples from user studies:

### Example 1
Query: "definition of a sigmet"
Retrieved Documents:
[1] [Score: 28.22] Definition of SIGMET: a notice of significant hazardous weather conditions (such as extreme turbulence, icing, or poor visibility) provided to aircraft pilots before takeoff.
[2] [Score: 25.97] SIGMET, or Significant Meteorological Information, is a weather advisory concerning the safety of all aircraft. There are two types: convective and non-convective.
[3] [Score: 25.51] SIGMET is a weather advisory for aircraft safety. Two types exist - convective and non-convective. Non-convective SIGMETs are issued for severe turbulence, severe icing, or instrument meteorological conditions.
[4] [Score: 25.15] SIGMET weather advisory with detailed criteria: severe turbulence over 3,000-square-mile area, severe icing over similar area, or IMC due to dust, sand, or volcanic ash.
[5] [Score: 24.22] Medical Definition of Sigmoid: the lower colon in human anatomy, shaped like the Greek letter sigma (C-shaped).

Analysis: The query seeks a technical aviation term definition. Top-4 documents are highly relevant and authoritative, all consistently defining SIGMET as a meteorological safety advisory for aircraft. These documents share common domain terminology (meteorological, aircraft, weather, safety, turbulence, icing). Doc-5 is a false match (medical term "sigmoid" vs aviation term "sigmet") but has low relevance score. The query intent is clear and unambiguous. Applying PRF can extract shared aviation/meteorological terminology (hazardous weather, aircraft safety, advisory, turbulence, visibility) to find more comprehensive explanations of aviation weather systems and related safety protocols.
Decision: YES

### Example 2
Query: "who is robert gray"
Retrieved Documents:
[1] [Score: 25.22] Robert Gray (1755-1806): American sea captain, first U.S. ship to circumnavigate the globe, explorer of Columbia River.
[2] [Score: 23.80] Captain Robert Gray discovered a great river and named it Columbia after his ship. His discovery formed basis of America's claim to Oregon Territory.
[3] [Score: 23.71] Robert Gray: Democratic candidate for Mississippi governor in 2015 elections, truck driver and former firefighter. Won primary but lost to incumbent Phil Bryant.
[4] [Score: 23.17] John Gray (born 1951): American relationship counselor, lecturer and author, associated with Maharishi Mahesh Yogi.
[5] [Score: 23.09] George Gray (born 1967): American game show host, announcer, and comedian from Missouri.

Analysis: Query about a person with multiple namesakes. Top-2 documents focus on the historical explorer Robert Gray (18th century maritime figure), while Doc-3 mentions a modern Mississippi politician with the same name. Docs-4,5 are false matches (different first names). Despite some ambiguity, the historical figure dominates the top results with high relevance. PRF can extract biographical and contextual terms (captain, explorer, Columbia River, maritime, circumnavigate, Oregon Territory) to strengthen retrieval of related historical exploration content and disambiguate from modern namesakes.
Decision: YES

### Example 3
Query: "what is theraderm used for"
Retrieved Documents:
[1] [Score: 23.62] Theraderm cream's main ingredient is lanolin from sheep's wool, used in skincare for boosting natural moisture barrier and drawing moisture to application site.
[2] [Score: 22.96] Theraderm is a manufacturer of clinical-grade skincare products: anti-aging creams, moisturizers, and acne system (Reversion brand). Founded 1996 by Dr. James Beckman.
[3] [Score: 21.86] A thermopile is a thermoelectric device consisting of thermocouples connected in series, widely used in non-contact temperature measurement and monitoring systems.
[4] [Score: 21.64] A theraband is a latex resistance band or tube used for physical therapy and light strength training, commonly used by athletes and dancers to strengthen feet.
[5] [Score: 21.59] A thermometer is a device used to measure temperature, used in healthcare to measure and monitor body temperature.

Analysis: The query seeks information about Theraderm skincare products. However, Top-5 documents show catastrophic topic drift. Only Docs-1,2 are relevant (skincare), while Docs-3,4,5 are completely unrelated - matching on similar spelling prefixes (thermo-/thera-) but covering physics and fitness domains. The result set is highly fragmented. Applying PRF would catastrophically mix terminology from these disparate domains (lanolin, thermocouples, resistance training, temperature), creating a nonsensical expanded query and causing massive query drift.
Decision: NO

### Example 4
Query: "causes of military suicide"
Retrieved Documents:
[1] [Score: 25.34] Canadian military study over 25 years: suicide was third leading cause of military deaths after motor vehicle accidents and cancer. Questions effectiveness of suicide prevention programs.
[2] [Score: 24.44] Suicide surpassed war as military's leading cause of death. War was the leading cause 2004-2011, but suicides became top cause of death for troops in 2012 and 2013.
[3] [Score: 24.29] Similar reporting: suicide surpassed war as leading cause of military death in 2012-2013 according to Pentagon medical statistical analysis.
[4] [Score: 24.21] WHO World Health Report 2004: deaths from intentional injuries (war, violence, suicide) were 2.8% of all deaths; unintentional injury was 6.2%.
[5] [Score: 24.14] Tragic anecdote about soldier Michael who saluted and shot himself. States suicide is now leading cause of death amongst active-duty soldiers.

Analysis: The query asks for "causes" (reasons/risk factors), but the retrieved documents primarily report "statistics" and "rankings" (leading cause of death, rates). The results are factually relevant to the topic but fail to address the specific "why" aspect of the user intent. They lack discussion of psychological factors, PTSD, or deployment stress. Applying PRF would likely extract statistical terms (leading cause, death rates, 2012-2013, fatalities) rather than explanatory causal terms. This would reinforce the focus on statistics rather than pivoting to the underlying causes the user requested.
Decision: NO

### Example 5
Query: "what is famvir prescribed for"
Retrieved Documents:
[1] [Score: 24.36] Famciclovir (Famvir) is a drug used for the treatment of genital herpes, cold sores, shingles, and chickenpox. Side effects, warnings, and precautions should be discussed.
[2] [Score: 24.14] Famciclovir (Famvir) is a prescription antiviral medication used to treat: Herpes simplex infections (cold sores and genital herpes) and Shingles.
[3] [Score: 23.94] Famciclovir (Famvir) is sometimes used to treat the herpes virus that causes cold sores and genital herpes, as well as the virus that causes shingles.
[4] [Score: 23.69] Famvir is approved by the FDA for the treatment of cold sores. Taking a single high dose can shorten the herpes infection.
[5] [Score: 23.48] Famciclovir (Famvir) is a prescription antiviral used to treat: Shingles in people with normal immune systems and recurring genital herpes.

Analysis: The query asks for the specific medical indications of Famvir. The top retrieved documents are excellent: they are highly consistent, authoritative, and directly list the exact conditions (genital herpes, cold sores, shingles, chickenpox). The user's information need is fully satisfied by the top results. Applying PRF in this case provides little marginal benefit and carries a risk of introducing unnecessary noise (e.g., specific side effects, dosage details, or generic antiviral terms) that could dilute the high precision of the current ranking. Since the answer is already saturated and precise, no expansion is needed.
Decision: NO
---

Now analyze this query step-by-step:

Query: "{query}"

Top-5 Documents:
{documents}

Think through these steps:

Step 1 - Query Intent: What is the user trying to find?

Step 2 - Result Quality: Are the top documents relevant and high-quality?

Step 3 - Topic Consistency: Do the documents share a coherent topic or is there drift?

Step 4 - PRF Impact: Will expanding with terms from these documents help or hurt?

Step 5 - Final Decision: Based on the above analysis, will PRF benefit or hurt?

Provide your response in this exact format:

Analysis:
[Your step-by-step reasoning here]

Decision: <YES or NO>

Confidence: <0-10 scale, where 0=completely uncertain, 10=absolutely certain>
"""

# ============ 加载数据 ============
print("=" * 80)
print("Chain-of-Thought PRF Prediction")
print("=" * 80)

print("\n1. 加载数据...")
df_preference = pd.read_csv('preference.csv')
df_queries = pd.read_csv('result_with_text.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')

if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()

eval_df = df_preference[df_preference['b_preference_ratio'] > 0].copy()
print(f"✓ 加载了 {len(eval_df)} 个有效查询")

# ============ 预测函数 (提取 YES/NO) ============
def predict_cot(qid, query, top5_docs, verbose=False):
    """
    使用 Chain-of-Thought，提取最终 YES/NO decision
    不依赖 logprobs，只看文本输出
    """
    
    docs_text = "\n".join([
        f"[{i+1}] [Score: {row['score']:.2f}] {str(row['passage_text'])[:200]}..."
        for i, (_, row) in enumerate(top5_docs.iterrows())
    ])
    
    prompt = COT_PROMPT_TEMPLATE.format(
        query=query,
        documents=docs_text
    )
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an IR expert. Think step-by-step."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,  # 稍微高一点，鼓励思考
            max_tokens=800
        )
        
        output = response.choices[0].message.content
        
        # 提取 Decision: YES/NO
        decision_match = re.search(r'Decision:\s*(YES|NO)', output, re.IGNORECASE)
        
        if decision_match:
            decision = decision_match.group(1).upper()
            prediction = "Benefit" if decision == "YES" else "Hurt"
        else:
            # Fallback: 看文本中哪个词出现更多
            yes_count = output.lower().count('yes') + output.lower().count('benefit')
            no_count = output.lower().count('no') + output.lower().count('hurt')
            
            if yes_count > no_count:
                decision = "YES"
                prediction = "Benefit"
            else:
                decision = "NO"
                prediction = "Hurt"
        
        # 提取 Confidence (0-10)
        confidence_match = re.search(r'Confidence:\s*(\d+)', output, re.IGNORECASE)
        
        if confidence_match:
            confidence_raw = int(confidence_match.group(1))
            # 转换 0-10 到 0-1
            confidence_normalized = confidence_raw / 10.0
        else:
            # 没有置信度，根据决定给默认值
            confidence_normalized = 0.8 if decision == "YES" else 0.2
        
        # 转换为 0-1 分数 (对应 b_preference_ratio)
        # YES (Benefit) → 0.5-1.0
        # NO (Hurt) → 0.0-0.5
        if decision == "YES":
            llm_score = 0.5 + confidence_normalized * 0.5  # 0.5 to 1.0
        else:
            llm_score = 0.5 - confidence_normalized * 0.5  # 0.0 to 0.5
        
        # 提取 Analysis
        analysis_match = re.search(r'Analysis:(.*?)Decision:', output, re.DOTALL | re.IGNORECASE)
        analysis = analysis_match.group(1).strip() if analysis_match else output[:200]
        
        if verbose:
            print(f"  Decision: {decision} → {prediction}")
            print(f"  Confidence: {confidence_raw if confidence_match else 'N/A'}/10")
            print(f"  LLM Score: {llm_score:.3f}")
        
        return {
            "prediction": prediction,
            "decision": decision,
            "confidence_raw": confidence_raw if confidence_match else None,
            "llm_score": llm_score,  # 0-1 连续分数
            "analysis": analysis,
            "full_output": output,
            "success": True
        }
        
    except Exception as e:
        if verbose:
            print(f"  Error: {e}")
        return {
            "prediction": "Hurt",
            "decision": "NO",
            "llm_score": 0.2,  # 默认分数
            "analysis": "",
            "full_output": "",
            "error": str(e),
            "success": False
        }

# ============ 批量预测 ============
print("\n2. 运行 CoT 预测...")
print("-" * 80)

results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Processing"):
    qid = row['qid']
    query = qid_to_query.get(qid, f"[Unknown QID {qid}]")
    
    top5 = df_colbert[df_colbert['qid'] == qid].nlargest(5, 'score')
    if len(top5) == 0:
        continue
    
    result = predict_cot(qid, query, top5, verbose=False)
    
    results.append({
        'qid': qid,
        'query': query,
        'true_b_pref_ratio': row['b_preference_ratio'],
        'true_preference': row['preference'],
        'llm_prediction': result['prediction'],
        'llm_decision': result['decision'],
        'llm_score': result.get('llm_score', 0.5),  # 连续分数
        'confidence_raw': result.get('confidence_raw', None),
        'analysis': result.get('analysis', ''),
        'success': result['success']
    })
    
    time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv('qwen_cot_predictions.csv', index=False, encoding='utf-8')
print(f"\n✓ 完成: {len(results_df)} queries, {results_df['success'].sum()} successful")

# ============ 评估 ============
eval_results = results_df[results_df['success'] == True].copy()

print("\n" + "=" * 80)
print("EVALUATION RESULTS (Chain-of-Thought)")
print("=" * 80)

print(f"\n有效样本数: {len(eval_results)}")

# 转换为数值用于相关性分析
# 使用 llm_score (0-1 连续值)
tau, p_tau = kendalltau(eval_results['llm_score'], eval_results['true_b_pref_ratio'])
rho, p_rho = spearmanr(eval_results['llm_score'], eval_results['true_b_pref_ratio'])

print(f"\nCorrelation Analysis (using continuous scores):")
print(f"  Kendall's Tau:  τ = {tau:.4f} (p = {p_tau:.4f})")
print(f"  Spearman's Rho: ρ = {rho:.4f} (p = {p_rho:.4f})")

sig = "***" if p_tau < 0.001 else ("**" if p_tau < 0.01 else ("*" if p_tau < 0.05 else ""))
strength = ("Negligible" if abs(tau) < 0.1 else 
            ("Weak" if abs(tau) < 0.3 else 
             ("Moderate" if abs(tau) < 0.5 else "Strong")))

direction = "Positive" if tau > 0 else "Negative"
print(f"  Correlation: {strength} {direction} {sig}")

# LLM Score 分布统计
print(f"\nLLM Score Distribution:")
print(f"  Mean:   {eval_results['llm_score'].mean():.3f}")
print(f"  Std:    {eval_results['llm_score'].std():.3f}")
print(f"  Min:    {eval_results['llm_score'].min():.3f}")
print(f"  Max:    {eval_results['llm_score'].max():.3f}")
print(f"  Median: {eval_results['llm_score'].median():.3f}")

# 如果是负相关，提示翻转
if tau < 0:
    print(f"\n⚠️  WARNING: Negative correlation detected!")
    print(f"  Model predictions are INVERTED.")
    print(f"  Consider flipping scores: llm_score_flipped = 1 - llm_score")
    
    # 计算翻转后的相关性
    eval_results['llm_score_flipped'] = 1 - eval_results['llm_score']
    tau_flip, p_flip = kendalltau(eval_results['llm_score_flipped'], 
                                   eval_results['true_b_pref_ratio'])
    print(f"\n  After flipping:")
    print(f"    Kendall's τ = {tau_flip:.4f} (p = {p_flip:.4f})")

# 分类性能
eval_results['true_label'] = eval_results['true_b_pref_ratio'].apply(
    lambda x: 'Benefit' if x > 0.55 else ('Hurt' if x < 0.45 else 'Neutral')
)
clf_df = eval_results[eval_results['true_label'] != 'Neutral'].copy()

if len(clf_df) > 0:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
    
    acc = accuracy_score(clf_df['true_label'], clf_df['llm_prediction'])
    f1 = f1_score(clf_df['true_label'], clf_df['llm_prediction'], 
                  pos_label='Benefit', zero_division=0)
    prec = precision_score(clf_df['true_label'], clf_df['llm_prediction'], 
                           pos_label='Benefit', zero_division=0)
    rec = recall_score(clf_df['true_label'], clf_df['llm_prediction'], 
                       pos_label='Benefit', zero_division=0)
    
    print(f"\nClassification Performance:")
    print(f"  Sample Size:  {len(clf_df)}")
    print(f"  Accuracy:     {acc:.4f}")
    print(f"  F1-Score:     {f1:.4f}")
    print(f"  Precision:    {prec:.4f}")
    print(f"  Recall:       {rec:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(clf_df['true_label'], clf_df['llm_prediction'], 
                         labels=['Benefit', 'Hurt'])
    print(f"\nConfusion Matrix:")
    print(f"                Predicted Benefit  Predicted Hurt")
    print(f"  True Benefit        {cm[0][0]:3d}               {cm[0][1]:3d}")
    print(f"  True Hurt           {cm[1][0]:3d}               {cm[1][1]:3d}")

# 预测分布
print(f"\nPrediction Distribution:")
print(f"  Predicted Benefit: {(eval_results['llm_prediction'] == 'Benefit').sum()}")
print(f"  Predicted Hurt:    {(eval_results['llm_prediction'] == 'Hurt').sum()}")
print(f"\nGround Truth Distribution:")
print(f"  True Benefit: {(clf_df['true_label'] == 'Benefit').sum()}")
print(f"  True Hurt:    {(clf_df['true_label'] == 'Hurt').sum()}")

# 对比 QPP
print(f"\n" + "=" * 80)
print("COMPARISON WITH BASELINES")
print("=" * 80)

print(f"\n{'Method':<20} {'Kendall τ':<12} {'P-value':<12} {'F1-Score':<12}")
print("-" * 60)
print(f"{'NQC':<20} {-0.1054:<12.4f} {0.3392:<12.4f} {'N/A':<12}")
print(f"{'WIG':<20} {0.0051:<12.4f} {0.9628:<12.4f} {'N/A':<12}")
print(f"{'DENSE_QPP':<20} {0.1260:<12.4f} {0.2534:<12.4f} {'N/A':<12}")
print("-" * 60)
print(f"{'Qwen CoT':<20} {tau:<12.4f} {p_tau:<12.4f} {f1:<12.4f}")
if tau < 0:
    print(f"{'Qwen CoT (flipped)':<20} {tau_flip:<12.4f} {p_flip:<12.4f} {'-':<12}")
print("-" * 60)

# 案例分析
print(f"\n" + "=" * 80)
print("SAMPLE PREDICTIONS")
print("=" * 80)

print("\n正确预测 Benefit:")
correct_benefit = clf_df[(clf_df['true_label'] == 'Benefit') & 
                         (clf_df['llm_prediction'] == 'Benefit')].head(2)
for _, row in correct_benefit.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    Analysis: {row['analysis'][:120]}...")
    print(f"    ✓ Correct: Benefit (true ratio={row['true_b_pref_ratio']:.3f})")

print("\n正确预测 Hurt:")
correct_hurt = clf_df[(clf_df['true_label'] == 'Hurt') & 
                      (clf_df['llm_prediction'] == 'Hurt')].head(2)
for _, row in correct_hurt.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    Analysis: {row['analysis'][:120]}...")
    print(f"    ✓ Correct: Hurt (true ratio={row['true_b_pref_ratio']:.3f})")

print("\n错误预测:")
incorrect = clf_df[clf_df['true_label'] != clf_df['llm_prediction']].head(2)
for _, row in incorrect.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    Analysis: {row['analysis'][:120]}...")
    print(f"    ✗ Wrong: Predicted {row['llm_prediction']}, True {row['true_label']} (ratio={row['true_b_pref_ratio']:.3f})")

# 总结
print(f"\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

print(f"""
🎯 Chain-of-Thought Performance:
   - Sample Size: {len(eval_results)}
   - Kendall's τ: {tau:.4f} ({strength} {direction}) {sig}
   - F1-Score: {f1:.4f}
   - Accuracy: {acc:.4f}
""")

if tau < -0.1 and p_tau < 0.05:
    print("⚠️  Model predictions are INVERTED (significant negative correlation)")
    print("   → Solution: Flip predictions (Benefit ↔ Hurt)")
    print(f"   → After flipping: τ = {tau_flip:.4f} (p = {p_flip:.4f})")
elif abs(tau) > 0.15 and p_tau < 0.05:
    print("✅ Model shows significant correlation with ground truth")
elif f1 > 0.65:
    print("✅ Good classification performance even without correlation")
else:
    print("⚠️  Results suggest model needs improvement")
    print("   → Try adding more diverse examples")
    print("   → Or analyze which query types it struggles with")

print(f"\n📁 Output: qwen_cot_predictions.csv")
print("=" * 80)

Chain-of-Thought PRF Prediction

1. 加载数据...
✓ 加载了 40 个有效查询

2. 运行 CoT 预测...
--------------------------------------------------------------------------------


Processing: 100%|██████████| 40/40 [13:16<00:00, 19.91s/it]


✓ 完成: 40 queries, 40 successful

EVALUATION RESULTS (Chain-of-Thought)

有效样本数: 40

Correlation Analysis (using continuous scores):
  Kendall's Tau:  τ = -0.0850 (p = 0.5219)
  Spearman's Rho: ρ = -0.1026 (p = 0.5289)
  Correlation: Negligible Negative 

LLM Score Distribution:
  Mean:   0.775
  Std:    0.219
  Min:    0.400
  Max:    0.900
  Median: 0.900

⚠️  WARNING: Negative correlation detected!
  Model predictions are INVERTED.
  Consider flipping scores: llm_score_flipped = 1 - llm_score

  After flipping:
    Kendall's τ = 0.0850 (p = 0.5219)

Classification Performance:
  Sample Size:  26
  Accuracy:     0.4231
  F1-Score:     0.5455
  Precision:    0.4500
  Recall:       0.6923

Confusion Matrix:
                Predicted Benefit  Predicted Hurt
  True Benefit          9                 4
  True Hurt            11                 2

Prediction Distribution:
  Predicted Benefit: 30
  Predicted Hurt:    10

Ground Truth Distribution:
  True Benefit: 13
  True Hurt:    13

COMPA